# 04 — Route-level Fairness Metrics từ `_front_details.txt`

Thực hiện **Section 7**: parse cấu trúc route chi tiết để tính các metric
fairness độc lập với Gini ($\rho_{max}$, CV, Jain index) cho các nghiệm
Distance-Focus đã chọn ở notebook 02.

> ⚠️ **QUAN TRỌNG**: Hàm `parse_front_details` trong `evrp_analysis_utils.py`
> dùng regex PLACEHOLDER vì chưa có file mẫu thật. Chạy cell 4.1 trên MỘT
> file `_front_details.txt` thật, in ra vài dòng đầu, rồi đối chiếu với các
> `RouteRecord` được parse ra. Nếu sai lệch, sửa các `_RE_*` trong module rồi
> `import importlib; importlib.reload(utils)` và chạy lại.


In [1]:
# ==== CẤU HÌNH ĐƯỜNG DẪN (chỉnh lại cho đúng máy của bạn) ====
import sys, os
sys.path.append(os.path.abspath("."))  # để import evrp_analysis_utils.py cùng thư mục

FULL_DIR  = "benchmark/full"     # thư mục kết quả bản đầy đủ (equity-aware)
NOEQ_DIR  = "benchmark/no-EQ"    # thư mục kết quả bản loại equity guidance
ARTIFACT_DIR = "artifacts"       # nơi lưu các bảng trung gian (csv) giữa các notebook
os.makedirs(ARTIFACT_DIR, exist_ok=True)

import pandas as pd
import numpy as np
import evrp_analysis_utils as utils

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)


In [2]:
td_solutions = pd.read_csv(f"{ARTIFACT_DIR}/td_solutions.csv")
manifest = pd.read_csv(f"{ARTIFACT_DIR}/run_manifest_matched_valid.csv")
td_solutions.head()


,SolutionID,Z1,Z2,Z3,Z4,IsFeasible,Variant,Instance,Seed,Z1_common
0,c101C10_s1_5,3,388.2454,0.1195,1162.8058,True,FULL,c101C10,1,3.0
1,c101C5_s1_1,2,257.7475,0.1065,872.0789,True,FULL,c101C5,1,2.0
2,c101_21_s7_31,12,1044.5000,0.0921,1199.3839,True,FULL,c101_21,7,12.0
3,c102_21_s2_18,11,1041.9537,0.0781,1215.2132,True,FULL,c102_21,2,11.0
4,c103C15_s3_11,3,374.4278,0.0668,1175.6675,True,FULL,c103C15,3,3.0


## 4.1. Kiểm thử parser trên MỘT file mẫu trước khi chạy hàng loạt

In [3]:
sample_run = manifest.iloc[0]
sample_path = os.path.join(
    sample_run["RunDir"], f"{sample_run['Instance']}_seed_{int(sample_run['Seed'])}_front_details.txt"
)

if os.path.exists(sample_path):
    with open(sample_path, encoding="utf-8", errors="ignore") as f:
        preview = "".join([next(f) for _ in range(25)])
    print("---- 25 dòng đầu file mẫu ----")
    print(preview)

    sample_routes = utils.parse_front_details(sample_path)
    print(f"\nSố RouteRecord parse được: {len(sample_routes)}")
    if sample_routes:
        print(sample_routes[0])
else:
    print(f"Không tìm thấy {sample_path} — kiểm tra lại FULL_DIR/NOEQ_DIR.")


---- 25 dòng đầu file mẫu ----
--- FINAL PARETO FRONT DETAILS ---

== Solution ID: 1
Solution (Vehicles: 12, Distance: 1060.81, WorkloadGini: 0.05, MaxTime: 1204.88, Feasible: true)
--- Route ID: 11 --- Feasible: YES
   Total Distance:       76.84
   Total Time:         1135.79
   Total Energy Cons:    76.84
--- Node Sequence (Count: 13) ---
  Idx | StrID |       Type |  ArrTime |  DepTime |   RemBat |  RemLoad |   Charge
---------------------------------------------------------------------------------------
    0 |    D0 |      Depot |     0.00 |     0.00 |    79.69 |   200.00 |     0.00
    1 |    S0 |    Station |     0.00 |     0.00 |    79.69 |   200.00 |     0.00
  111 |   C90 |   Customer |    20.62 |   111.00 |    59.07 |   190.00 |     0.00
  108 |   C87 |   Customer |   116.00 |   206.00 |    54.07 |   170.00 |     0.00
  107 |   C86 |   Customer |   207.00 |   297.00 |    53.07 |   160.00 |     0.00
  103 |   C82 |   Customer |   306.00 |   549.00 |    44.07 |   140.00 |    

### -> Nếu số RouteRecord = 0 hoặc field sai, chỉnh regex `_RE_SOLUTION_HEADER`, `_RE_VEHICLE_HEADER`, `_RE_CUSTOMER`, `_RE_STATION`, `_RE_TIME_FIELD` trong `evrp_analysis_utils.py` cho khớp định dạng thật, rồi chạy lại cell trên.

## 4.2. Parse route details cho các nghiệm TD đã chọn (theo instance, cả hai variant)

Giả định: nghiệm TD nằm trong file `_front_details.txt` của RunDir tương ứng seed đã đóng góp nghiệm đó.
Nếu cột `Seed` không có trong `td_solutions`, cần parse toàn bộ archive của instance/variant đó.

In [4]:
all_workload_rows = []
for _, row in td_solutions.iterrows():
    variant, instance = row["Variant"], row["Instance"]
    seed = row.get("Seed")
    cand_runs = manifest[(manifest["Variant"] == variant) & (manifest["Instance"] == instance)]
    if seed is not None and not pd.isna(seed):
        cand_runs = cand_runs[cand_runs["Seed"] == seed]

    for _, run in cand_runs.iterrows():
        path = os.path.join(run["RunDir"], f"{run['Instance']}_seed_{int(run['Seed'])}_front_details.txt")
        routes = utils.parse_front_details(path)
        if not routes:
            continue
        wl = utils.routes_to_workload_df(routes)
        wl["Variant"] = variant
        wl["Instance"] = instance
        wl["Seed"] = run["Seed"]
        all_workload_rows.append(wl)

workload_all = pd.concat(all_workload_rows, ignore_index=True) if all_workload_rows else pd.DataFrame()
workload_all.to_csv(f"{ARTIFACT_DIR}/route_workload_raw.csv", index=False)
print(f"{len(workload_all)} route-level rows parsed")
workload_all.head()


29706 route-level rows parsed


,SolutionID,VehicleID,NumCustomers,NumStations,TravelTime,ServiceTime,ChargeTime,WaitTime,ActiveWorkload_A,TotalTime_H,Variant,Instance,Seed
0,1,0,3,3,1044.19,0.0,0.0,0.0,1044.19,1044.19,FULL,c101C10,1
1,1,0,4,2,1162.81,0.0,0.0,0.0,1162.81,1162.81,FULL,c101C10,1
2,1,2,3,2,1119.16,0.0,0.0,0.0,1119.16,1119.16,FULL,c101C10,1
3,2,2,3,2,1119.16,0.0,0.0,0.0,1119.16,1119.16,FULL,c101C10,1
4,2,0,4,2,1162.81,0.0,0.0,0.0,1162.81,1162.81,FULL,c101C10,1


## 4.3. Tính $\rho_{max}$, CV, Jain index, Gini mỗi (Variant, Instance, Seed, SolutionID)

In [5]:
if not workload_all.empty:
    fairness = utils.route_fairness_metrics(
        workload_all, group_cols=("Variant", "Instance", "Seed", "SolutionID")
    )
    fairness.to_csv(f"{ARTIFACT_DIR}/route_fairness_metrics.csv", index=False)
    print(fairness.groupby("Variant")[["Gini", "rho_max", "CV", "Jain"]].mean())
    fairness.head(10)
else:
    fairness = pd.DataFrame()
    print("Chưa có dữ liệu route-level — kiểm tra lại parser ở 4.1 trước.")


             Gini   rho_max       CV      Jain
Variant                                       
FULL     0.030996  1.067510  0.05908  0.994168
No-EQ    0.032324  1.064873  0.06277  0.991651


## 4.4. Safeguard: kiểm tra fairness gain không đến từ 'padding' charging time

So sánh $A_{sum}$, $W_{sum}$, ChargeTime\_sum, NumStations giữa FULL và No-EQ.

In [6]:
if not fairness.empty:
    safeguard = fairness.groupby("Variant")[["A_sum", "W_sum", "ChargeTime_sum", "NumStations"]].mean()
    print(safeguard)


               A_sum  W_sum  ChargeTime_sum  NumStations
Variant                                                 
FULL     4492.271860    0.0             0.0    13.197516
No-EQ    4306.085107    0.0             0.0     8.159844
